# 03 LiDAR Velocity

This notebook estimates LiDAR target velocity using the prepared dataset outputs from `01_dataset_preparation.ipynb`.

The original LiDAR velocity analysis logic from `condition_velocity.ipynb` is retained.

The final LiDAR velocity table is saved to `results/condition/` for use by later analysis notebooks.

In [1]:
import sys
sys.path.append("../..")

from pathlib import Path

import pandas as pd
from truckscenes import TruckScenes

from src.config import DATA_ROOT, SENSOR_ROOT
from src.lidar_velocity import (
    build_target_lidar_information_df,
    build_lidar_target_states_df,
    build_lidar_velocity_df,
)

In [2]:
RESULT_DIR = Path("../../results/condition")

## 1. Load Prepared Data

Load the dataset preparation outputs generated by `01_dataset_preparation.ipynb`.

In [ ]:
available_sample_data = pd.read_csv(RESULT_DIR / "available_sample_data.csv")
lidar_observation_df = pd.read_csv(RESULT_DIR / "lidar_observation_df.csv")

In [4]:
trucksc = TruckScenes(dataroot=DATA_ROOT)

Auto-detected latest version: v1.2-mini
Loading truckscenes tables for version v1.2-mini...
11 attribute,
18 calibrated_sensor,
27 category,
20090 ego_motion_cabin,
20089 ego_motion_chassis,
20116 ego_pose,
1094 instance,
400 sample,
25750 sample_annotation,
43556 sample_data,
10 scene,
18 sensor,
4 visibility,
10 weather_annotation,
Done loading in 0.738 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


f:\CITS3200\Boeing-RADAR-Autonomous-\.venv311\Lib\site-packages\truckscenes\truckscenes.py:100: UserWarning: The visualization dependencies are not installed on your system! Run 'pip install "truckscenes-devkit[all]"'.
  warnings.warn('''The visualization dependencies are not installed on your system! '''


## 2. Build LiDAR Velocity

In [ ]:
lidar_tokens = set(lidar_observation_df["Token"])
lidar_sample_tokens = set(lidar_observation_df["Sample Token"])
lidar_sample_data = available_sample_data[available_sample_data["token"].isin(lidar_tokens)].copy()
lidar_sample_data_lookup = lidar_sample_data.set_index("token").to_dict("index")

ego_pose_tokens = set(lidar_sample_data["ego_pose_token"].dropna())
lidar_ego_pose_lookup = {
    x["token"]: x
    for x in trucksc.ego_pose
    if x["token"] in ego_pose_tokens
}

calibrated_sensor_tokens = set(lidar_sample_data["calibrated_sensor_token"].dropna())
lidar_calibrated_sensor_lookup = {
    x["token"]: x
    for x in trucksc.calibrated_sensor
    if x["token"] in calibrated_sensor_tokens
}

lidar_annotations = [
    x
    for x in trucksc.sample_annotation
    if x["sample_token"] in lidar_sample_tokens
]

In [6]:
target_lidar_df = build_target_lidar_information_df(
    lidar_annotations,
    trucksc.instance,
    trucksc.category,
)

lidar_target_states_df = build_lidar_target_states_df(
    target_lidar_df,
    lidar_annotations,
    lidar_observation_df,
    lidar_sample_data_lookup,
    lidar_ego_pose_lookup,
    lidar_calibrated_sensor_lookup,
    SENSOR_ROOT,
)

lidar_velocity_df = build_lidar_velocity_df(
    lidar_target_states_df,
    lidar_annotations,
)

print("LiDAR velocity intervals:", len(lidar_velocity_df))

LiDAR velocity intervals: 8007


## 3. Save LiDAR Velocity

In [ ]:
lidar_velocity_df.to_csv(RESULT_DIR / "lidar_velocity_df.csv", index=False)
print("LiDAR velocity results saved.")

LiDAR velocity results saved.
